## Feature Engineering and Data Selection for HeRG Dataset

In [ ]:
from src.core.fingerprints import Fingerprints
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from src.core.utils import select_diverse_complex_subset

THR = 0.99

def visualize_target(df: pd.DataFrame, target_name: str, n_bins: int = 50) -> None:
    fig, (ax_box, ax_hist) = plt.subplots(
        2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)}, figsize=(10, 8)
    )
    sns.boxplot(x=df[target_name], ax=ax_box, orient='h')
    ax_box.set(xlabel='')
    sns.histplot(df[target_name], bins=n_bins, kde=True, ax=ax_hist)

    ax_box.set_title(f'Distribution of {target_name}')
    plt.show()

def print_stats(df: pd.DataFrame, target_name: str) -> None:
    print(f"Mean: {df[target_name].mean():.4f}")
    print(f"Median: {df[target_name].median():.4f}")
    print(f"Standard Deviation: {df[target_name].std():.4f}")
    print(f"Minimum: {df[target_name].min():.4f}")
    print(f"Maximum: {df[target_name].max():.4f}")

def ecfp_features(smiles_name: str, df: pd.DataFrame, threshold: float = 0.99) -> pd.DataFrame:
    names = ['ecfp']
    params = {
        'ecfp': {'radius': 2, 'size': 1024, 'count': True},
    }
    fingerprints, f_names = Fingerprints().apply(
        smiles=df[smiles_name].tolist(),
        names=names,
        **params
    )
    df_fingerprints = pd.DataFrame(fingerprints, columns=f_names)
    scaler = MinMaxScaler()
    df_features_scaled = scaler.fit_transform(df_fingerprints)
    selector = VarianceThreshold(threshold=1 - threshold)
    selector.fit(df_features_scaled)
    df_fingerprints = df_fingerprints.iloc[:, selector.get_support(indices=True)]
    df_fingerprints[smiles_name] = df[smiles_name].values
    return df_fingerprints

In [ ]:
df = pd.read_csv('../data/raw/herg.csv')

smiles = df['SMILES'].to_numpy()
smiles

In [ ]:
from rdkit import Chem

mols = [Chem.MolFromSmiles(s) for s in smiles]
selected_molecules = select_diverse_complex_subset(mols, num_to_select=500, similarity_cutoff=0.3)

df = pd.DataFrame({'smiles': [s for i,s in enumerate(smiles) if i in selected_molecules]})

In [ ]:
df

In [ ]:
df_fingerprints = ecfp_features('smiles', df, threshold=THR)
df_fingerprints

In [ ]:
df_fingerprints = df_fingerprints.drop_duplicates(subset=[c for c in df_fingerprints.columns if c != 'smiles']).reset_index(drop=True)
df_fingerprints

In [ ]:
df_fingerprints.to_csv('../data/herg_data/herg_ecfp.csv', index=False)

In [ ]:
df_fingerprints['non_zero_count'] = df_fingerprints.drop(columns=['smiles']).sum(axis=1) + df_fingerprints.drop(columns=['smiles']).astype(bool).sum(axis=1)

# Sort the DataFrame by the non-zero count in descending order and select the top rows
# The 'mergesort' kind is used for stable sorting
top_rows = df_fingerprints.sort_values(by='non_zero_count', ascending=False, kind='mergesort')
df_fingerprints.drop(columns=['non_zero_count'], inplace=True)
top_rows['non_zero_count']

## Selecting potential features for target functions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

potential_features = df_fingerprints[[col for col in df_fingerprints.columns if df_fingerprints[col].value_counts(normalize=True).iloc[0] <= 0.6 and col != 'smiles']]

ncols = 3
nrows = math.ceil(len(potential_features.columns) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()
for i, feature in enumerate(potential_features.columns):
    sns.histplot(data=potential_features, x=feature, kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Distribution of {feature}', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency', fontsize=10)
for j in range(len(potential_features.columns), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.suptitle('Distribution of Individual Features', y=1.0, fontsize=18)
plt.show()

## Select "most independent" bits

In [ ]:
candidate_bits = [int(c.split('_')[-1]) for c in potential_features.columns]
mols = [Chem.MolFromSmiles(s) for s in df_fingerprints['smiles'].values]
candidate_bits

In [ ]:
import numpy as np
from rdkit.Chem import AllChem

def get_only_nonzero_radius_candidates(mols):
    """
    Scans the dataset and returns only bits that are
    specifically generated at Radius 2.
    """
    smallest_radius = {i: np.inf for i in range(0, 1024)}
    for mol in mols:
        bit_info = {}
        _ = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024, bitInfo=bit_info, useChirality=True)

        for bit_id, occurrences in bit_info.items():
            for info in occurrences:
                rad = info[1]
                if rad < smallest_radius[bit_id]:
                    smallest_radius[bit_id] = rad

    radius_bits = [i for i, value in smallest_radius.items() if value != 0]

    return list(radius_bits)

non_zero_bits = get_only_nonzero_radius_candidates(mols)
candidate_bits = [c for c in candidate_bits if c in non_zero_bits]
candidate_bits

In [ ]:
from collections import defaultdict
from rdkit.Chem import Draw, AllChem


def visualize_bits_in_dataset(mols, bit_list, mols_per_bit=3, radius=2):
    """
    Visualizes the selected independent bits by showing examples
    from the dataset where they occur.
    """
    # 1. Map bits to molecules where they appear
    bit_occurrences = defaultdict(list)
    dataset_frequencies = defaultdict(int)

    for mol in mols:
        bi = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=1024, bitInfo=bi, useChirality=True)
        on_bits = fp.GetOnBits()

        for bit_id in bit_list:
            if bit_id in on_bits:
                dataset_frequencies[bit_id] += 1
                # Save first few examples for visualization
                if len(bit_occurrences[bit_id]) < mols_per_bit:
                    # RDKit DrawMorganBits expects (mol, bit_id, bit_info)
                    bit_occurrences[bit_id].append((mol, bit_id, bi))

    # 2. Prepare the grid drawing
    draw_items = []
    legends = []

    for bit_id in bit_list:
        examples = bit_occurrences[bit_id]
        freq = dataset_frequencies[bit_id]

        for i, (mol, b_id, bi) in enumerate(examples):
            draw_items.append((mol, b_id, bi))
            # Label: Bit ID, Dataset Frequency, and Example #
            legends.append(f"Bit {b_id} (Freq: {freq})\nExample {i+1}")

    # 3. Render the grid
    # This renders the bit environments highlighted within their source molecules
    img = Draw.DrawMorganBits(
        draw_items,
        molsPerRow=mols_per_bit,
        subImgSize=(300, 300),
        legends=legends,
        useSVG=True
    )

    return img

mols = [Chem.MolFromSmiles(s) for s in df_fingerprints['smiles'].values]
visualize_bits_in_dataset(mols, candidate_bits)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from collections import defaultdict

def get_bit_atom_indices(mol, bit_id, boi, radius=2, n_bits=1024):
    """
    Returns a list of sets. Each set contains the atom indices
    for one occurrence of the bit_id in the molecule.
    """
    bi = {}
    _ = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits, bitInfo=bi, useChirality=True)
    if bit_id not in bi:
        return []

    occurrence_atoms = []
    for center, rad in bi[bit_id]:
        if rad == 0:
            occurrence_atoms.append({center})
        else:
            env_path = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, center)
            atoms = set()
            for b_idx in env_path:
                bond = mol.GetBondWithIdx(b_idx)
                atoms.update([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
            atoms.add(center)
            occurrence_atoms.append(atoms)
    return occurrence_atoms

def select_relaxed_islands(mols, candidate_bits, boi, max_overlap=0.1, max_corr=0.2):
    """
    Selects bits that are 'mostly' independent.
    - max_overlap: Max allowed percentage of shared atoms between two bits.
    - max_corr: Max statistical correlation between bit presence.
    """
    # 1. Generate Bit Matrix for Correlation check
    n_bits = 1024
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n_bits, useChirality=True) for m in mols]
    bit_vectors = {}
    for b_id in candidate_bits:
        vec = np.array([fp.GetBit(b_id) for fp in fps])
        if vec.sum() > 0: # Ensure bit actually exists in sample
            bit_vectors[b_id] = vec

    # 2. Pre-calculate Bit Atom-Usage
    # bit_to_mol_atoms[bit_id][mol_idx] = set of atom indices
    bit_to_mol_atoms = defaultdict(dict)
    for b_id in bit_vectors.keys():
        print(b_id)
        for i, m in enumerate(mols):
            indices = get_bit_atom_indices(m, b_id, boi)
            if indices:
                # We take the union of all occurrences in that mol for the check
                bit_to_mol_atoms[b_id][i] = set().union(*indices)

    final_bits = []
    sorted_candidates = sorted(bit_vectors.keys(), key=lambda b: bit_vectors[b].sum(), reverse=True)

    for b_id in sorted_candidates:
        is_independent = True
        b_vec = bit_vectors[b_id]
        b_atoms = bit_to_mol_atoms[b_id]

        for accepted_id in final_bits:
            corr = np.corrcoef(b_vec, bit_vectors[accepted_id])[0, 1]
            if corr > max_corr:
                is_independent = False
                break

            overlap_scores = []
            common_mols = set(b_atoms.keys()).intersection(set(bit_to_mol_atoms[accepted_id].keys()))

            for m_idx in common_mols:
                set1 = b_atoms[m_idx]
                set2 = bit_to_mol_atoms[accepted_id][m_idx]

                intersection = len(set1.intersection(set2))
                union = len(set1.union(set2))
                jaccard = intersection / union if union > 0 else 0
                overlap_scores.append(jaccard)

            if overlap_scores and np.mean(overlap_scores) > max_overlap:
                is_independent = False
                break

        if is_independent:
            final_bits.append(b_id)
            print(f"Accepted Relaxed Island: {b_id} (Freq: {b_vec.sum()})")

    return final_bits

In [ ]:
considered_bits = [int(c.split('_')[-1]) for c in df_fingerprints.columns if 'ecfp' in c]
final_bits = select_relaxed_islands(mols, candidate_bits, considered_bits)

In [ ]:
visualize_bits_in_dataset(mols, final_bits)

In [ ]:
final_features = potential_features[[f'ecfp_feature_{i}' for i in final_bits]]

ncols = 3
nrows = math.ceil(len(final_features.columns) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()
for i, feature in enumerate(final_features.columns):
    sns.histplot(data=final_features, x=feature, kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Distribution of {feature}', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency', fontsize=10)
for j in range(len(final_features.columns), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.suptitle('Distribution of Individual Features', y=1.0, fontsize=18)
plt.show()

In [ ]:
final_features.corr()

## Synthetic functions

In [ ]:
df = pd.read_csv('../data/raw/herg_ecfp.csv')

selected_features = [726, 456, 893, 428]
f_names = [f'ecfp_feature_{i}' for i in selected_features]

plt.figure(figsize=(len(selected_features) * 4, len(selected_features) * 4))
pair_plot = sns.pairplot(df[f_names], kind='kde')

# Add a title for the entire figure.
pair_plot.fig.suptitle('Pairwise Relationships of Selected Features', y=1.02, fontsize=16)

# Display the plot.
plt.show()

In [ ]:
from itertools import combinations

feature_pairs = list(combinations(f_names, 2))
ncols = 3
nrows = math.ceil(len(feature_pairs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()
for i, (f1, f2) in enumerate(feature_pairs):
    product = df[f1] * df[f2]
    sns.histplot(product, kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Product of {f1} & {f2}', fontsize=12)
    axes[i].set_xlabel('Product Value')
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(len(feature_pairs), len(axes)):
    axes[j].set_visible(False)

# Adjust layout and add a main title
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.suptitle('Distribution of Pairwise Feature Products', fontsize=18)
plt.show()

In [ ]:
selected_features = [726, 456, 893, 428]
f_726 = df[f'ecfp_feature_{selected_features[0]}']
f_456 = df[f'ecfp_feature_{selected_features[1]}']
f_893 = df[f'ecfp_feature_{selected_features[2]}']
f_428 = df[f'ecfp_feature_{selected_features[3]}']

for v in f_893.unique():
    indices = np.where(f_893 == v)
    subset_726 = f_726.iloc[indices]
    subset_456 = f_456.iloc[indices]
    subset_428 = f_428.iloc[indices]

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    sns.histplot(subset_726, kde=True, bins=20, ax=ax[0])
    sns.histplot(subset_456, kde=True, bins=20, ax=ax[1])
    sns.histplot(subset_428, kde=True, bins=20, ax=ax[2])

    ax[0].title.set_text(f'726 value {v}')
    ax[1].title.set_text(f'456 value {v}')
    ax[2].title.set_text(f'428 value {v}')

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
import copy
import os
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

selected_features = [726, 456, 893, 428]

def simple_linear_function4(df: pd.DataFrame) -> pd.Series:
    f_726 = df[f'ecfp_feature_{selected_features[0]}'].values
    f_456 = df[f'ecfp_feature_{selected_features[1]}'].values
    f_893 = df[f'ecfp_feature_{selected_features[2]}'].values
    f_428 = df[f'ecfp_feature_{selected_features[3]}'].values

    target = 7.5 * f_726 + 5 * f_428 + 5.5 * f_893 - 1.5 * f_456
    return pd.Series(target)

def piecewise_linear_function4(df: pd.DataFrame) -> pd.Series:
    f_726 = df[f'ecfp_feature_{selected_features[0]}'].values.flatten()
    f_456 = df[f'ecfp_feature_{selected_features[1]}'].values.flatten()
    f_893 = df[f'ecfp_feature_{selected_features[2]}'].astype(int).values.flatten()
    f_428 = df[f'ecfp_feature_{selected_features[3]}'].values.flatten()

    conditions = [
        (f_893 <= 0),
        (f_893 == 1),
        (f_893 >= 2)
    ]

    choices = [
        5.5 * f_726 + 1.5 * f_428,
        1.5 * f_456 - 2.5 * f_726,
        -1.5 * f_428
    ]

    target = np.select(conditions, choices, default=-1000.0)

    if (target == -1000.0).any():
        problem_values = f_893[target == -1]
        print(f"Values in f_893 that failed all conditions: {np.unique(problem_values)}")

    return pd.Series(target, index=df.index)

def nonlinear_function4(df: pd.DataFrame) -> pd.Series:
    f_726 = df[f'ecfp_feature_{selected_features[0]}'].values
    f_456 = df[f'ecfp_feature_{selected_features[1]}'].values
    f_893 = df[f'ecfp_feature_{selected_features[2]}'].values
    f_428 = df[f'ecfp_feature_{selected_features[3]}'].values

    component1 = 4.5 * f_456 + 2.5 * f_893
    component2 = 3 * f_726 * f_428
    component3 = -1.5 * f_456 * f_726
    target = component1 + component2 + component3
    return pd.Series(target)

df = pd.read_csv('../data/raw/herg_ecfp.csv')
X = df.drop('smiles', axis=1)

functions = {
    'linear': simple_linear_function4,
    'piecewise': piecewise_linear_function4,
    'nonlinear': nonlinear_function4,
}

for name, func in functions.items():
    y = func(X)

    print(f"\nDescriptive Statistics for {name}:")
    print(y.describe())

    df_current = copy.deepcopy(df)
    df_current['target'] = y

    print("threshold: ", round((df_current['target'].max() - df_current['target'].min()) * 1/10, 1))

    df_current.to_csv(f"../data/herg_data/herg_ecfp_{name}.csv", index=False)

    plt.figure(figsize=(10, 6))
    sns.histplot(y, kde=True, bins=40)
    plt.title(f'Distribution of {name}', fontsize=16)
    plt.xlabel('Target Value', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(axis='y', alpha=0.5)
    plt.show()
    print('=' * 50)